# **Assignment 5**


**Anja Pijak**  
ap72051@student.uni-lj.si


In [21]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
data = pd.read_csv("datasets/wine_quality.csv")

data = data.iloc[:, 2:]
data['alcohol_level'] = data['alcohol_level'].apply(lambda x: 1 if x == "high" else 0)
data['quality'] = data['quality'].apply(lambda x: 1 if x == "high" else 0)

# Split features and target
X = data.drop('quality', axis=1)
y = data['quality']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [3]:
data.head()

,fixed.acidity,volatile.acidity,citric.acid,residual.sugar,chlorides,free.sulfur.dioxide,total.sulfur.dioxide,density,pH,sulphates,quality,alcohol_level
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,0,0
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,0,0
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,0,1
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,0,0
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,0,0


## Gradient boosting implementation

In [4]:
# Gradient Boosting 
class GradientBoosting:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.initial_prediction = None

    def fit(self, X, y):
        self.initial_prediction = y.mode()[0]
        residual = y.apply(lambda v: 1 if v != self.initial_prediction else 0)  # Initialize residuals

        for _ in range(self.n_estimators):
            # Fit on residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42)
            tree.fit(X, residual)
            predictions = tree.predict(X)

            # Update residuals
            residual -= self.learning_rate * predictions

            self.trees.append(tree)

    def predict(self, X):
        # Start with the initial prediction (constant value)
        predictions = np.full(X.shape[0], self.initial_prediction, dtype=float)
        # Add contributions from each tree
        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)

        return np.round(predictions).astype(int)  # Round to get 0 or 1 predictions


In [5]:
# Train and predictions
gb_model = GradientBoosting(n_estimators=50, learning_rate=0.1, max_depth=2)
gb_model.fit(X_train, y_train)


y_pred_train = gb_model.predict(X_train)
y_pred_test = gb_model.predict(X_test)

In [6]:
# Accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.2f}")
print(f"Test Accuracy: {test_accuracy:.2f}")

Training Accuracy: 0.82
Test Accuracy: 0.79


### Testing parameters

Next few sections are testing the performance with different parameters

In [ ]:
from tabulate import tabulate

results = []

# Evaluate performance with different tree heights
for depth in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:

    gb_model = GradientBoosting(n_estimators=50, learning_rate=0.1, max_depth=depth)
    gb_model.fit(X_train, y_train)
    
    # Train accuracy
    y_pred_train = gb_model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_pred_train) 
    # Test accuracy
    y_pred_test = gb_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    
    results.append([depth, train_accuracy, test_accuracy])

headers = ["Max Depth", "Train Accuracy", "Test Accuracy"]
print(tabulate(results, headers=headers, tablefmt="grid"))


+-------------+------------------+-----------------+
|   Max Depth |   Train Accuracy |   Test Accuracy |
+=============+==================+=================+
|           1 |         0.787647 |        0.766327 |
+-------------+------------------+-----------------+
|           2 |         0.822103 |        0.793878 |
+-------------+------------------+-----------------+
|           3 |         0.843798 |        0.811224 |
+-------------+------------------+-----------------+
|           4 |         0.873915 |        0.829592 |
+-------------+------------------+-----------------+
|           5 |         0.905054 |        0.846939 |
+-------------+------------------+-----------------+
|           6 |         0.940786 |        0.85102  |
+-------------+------------------+-----------------+
|           7 |         0.971669 |        0.860204 |
+-------------+------------------+-----------------+
|           8 |         0.990812 |        0.873469 |
+-------------+------------------+------------

For tree depth, we can see from the table that it starts to overfit very soon, after depth 5 it is very obvious, but for depths 4 and 5 we see noticable difference in train and test set accuracy.

In [19]:
resultslr = []

# Evaluate performance with different tree heights
for depth in [3]:
    for lr in [0.0001, 0.001, 0.01, 0.1, 0.2, 0.5, 1, 1.5]:
        gb_model = GradientBoosting(n_estimators=50, learning_rate=lr, max_depth=depth)
        gb_model.fit(X_train, y_train)
        
        # Train accuracy
        y_pred_train = gb_model.predict(X_train)
        train_accuracy = accuracy_score(y_train, y_pred_train) 
        # Test accuracy
        y_pred_test = gb_model.predict(X_test)
        test_accuracy = accuracy_score(y_test, y_pred_test)
        
        
        resultslr.append([lr, train_accuracy, test_accuracy])

headers = ["Learning Rate", "Train Accuracy", "Test Accuracy"]
print(tabulate(resultslr, headers=headers, tablefmt="grid"))


+-----------------+------------------+-----------------+
|   Learning Rate |   Train Accuracy |   Test Accuracy |
+=================+==================+=================+
|          0.0001 |         0.787392 |        0.768367 |
+-----------------+------------------+-----------------+
|          0.001  |         0.787392 |        0.768367 |
+-----------------+------------------+-----------------+
|          0.01   |         0.787392 |        0.768367 |
+-----------------+------------------+-----------------+
|          0.1    |         0.843798 |        0.811224 |
+-----------------+------------------+-----------------+
|          0.2    |         0.867279 |        0.830612 |
+-----------------+------------------+-----------------+
|          0.5    |         0.904033 |        0.837755 |
+-----------------+------------------+-----------------+
|          1      |         0.918326 |        0.808163 |
+-----------------+------------------+-----------------+
|          1.5    |         0.9

From the table above we see overfitting starts when lr>0.02.

In [40]:
resultnsest = []

# Evaluate performance with different tree heights

for n_estimators in [10,30,50,70,100,130, 150]:
    gb_model = GradientBoosting(n_estimators, learning_rate=0.2, max_depth=3)
    gb_model.fit(X_train, y_train)
    
    # Train accuracy
    y_pred_train = gb_model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_pred_train) 
    # Test accuracy
    y_pred_test = gb_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    
    resultnsest.append([n_estimators, train_accuracy, test_accuracy])

headers = ["Number of Estimators", "Train Accuracy", "Test Accuracy"]
print(tabulate(resultnsest, headers=headers, tablefmt="grid"))


+------------------------+------------------+-----------------+
|   Number of Estimators |   Train Accuracy |   Test Accuracy |
+========================+==================+=================+
|                     10 |         0.816233 |        0.789796 |
+------------------------+------------------+-----------------+
|                     30 |         0.85707  |        0.822449 |
+------------------------+------------------+-----------------+
|                     50 |         0.867279 |        0.830612 |
+------------------------+------------------+-----------------+
|                     70 |         0.877999 |        0.822449 |
+------------------------+------------------+-----------------+
|                    100 |         0.89612  |        0.82449  |
+------------------------+------------------+-----------------+
|                    130 |         0.90633  |        0.835714 |
+------------------------+------------------+-----------------+
|                    150 |         0.912

Then for number of estimators we see the results don't change as fast as when we checked for depth and lr. However, it doesn't seem that we should take number of estimators between 50 and 70.

From all the results in testing different parameters, the optimal parameters to prevent overfitting are depth 3-4, lr 0.2 and number of trees around 50.

## Comparing with scikit

In [33]:
from tabulate import tabulate
from sklearn.metrics import classification_report, accuracy_score

# Train custom model
gb_model = GradientBoosting(n_estimators=50, learning_rate=0.2, max_depth=3)
gb_model.fit(X_train, y_train)

# Train scikit-learn model
sklearn_gb_model = GradientBoostingClassifier(n_estimators=50, learning_rate=0.2, max_depth=3, random_state=42)
sklearn_gb_model.fit(X_train, y_train)

# Predictions
y_pred_train_custom = gb_model.predict(X_train)
y_pred_test_custom = gb_model.predict(X_test)

y_pred_train_sklearn = sklearn_gb_model.predict(X_train)
y_pred_test_sklearn = sklearn_gb_model.predict(X_test)

# Classification reports in table format
custom_train_report = classification_report(y_train, y_pred_train_custom, output_dict=True)
custom_test_report = classification_report(y_test, y_pred_test_custom, output_dict=True)

sklearn_train_report = classification_report(y_train, y_pred_train_sklearn, output_dict=True)
sklearn_test_report = classification_report(y_test, y_pred_test_sklearn, output_dict=True)

# Extract precision, recall, F1, and accuracy
def extract_metrics(report, model_name, dataset_type, y_true, y_pred):
    metrics = report['weighted avg']
    accuracy = accuracy_score(y_true, y_pred)
    return [model_name, dataset_type, metrics['precision'], metrics['recall'], metrics['f1-score'], accuracy]

summary_table = [
    extract_metrics(custom_train_report, "Custom Model", "Train", y_train, y_pred_train_custom),
    extract_metrics(custom_test_report, "Custom Model", "Test", y_test, y_pred_test_custom),
    extract_metrics(sklearn_train_report, "Scikit-learn Model", "Train", y_train, y_pred_train_sklearn),
    extract_metrics(sklearn_test_report, "Scikit-learn Model", "Test", y_test, y_pred_test_sklearn),
]

print(tabulate(summary_table, headers=["Model", "Dataset", "Precision", "Recall", "F1-Score", "Accuracy"], tablefmt="grid"))


+--------------------+-----------+-------------+----------+------------+------------+
| Model              | Dataset   |   Precision |   Recall |   F1-Score |   Accuracy |
+====================+===========+=============+==========+============+============+
| Custom Model       | Train     |    0.863225 | 0.867279 |   0.852865 |   0.867279 |
+--------------------+-----------+-------------+----------+------------+------------+
| Custom Model       | Test      |    0.82028  | 0.830612 |   0.811443 |   0.830612 |
+--------------------+-----------+-------------+----------+------------+------------+
| Scikit-learn Model | Train     |    0.866366 | 0.871873 |   0.86087  |   0.871873 |
+--------------------+-----------+-------------+----------+------------+------------+
| Scikit-learn Model | Test      |    0.810147 | 0.823469 |   0.80535  |   0.823469 |
+--------------------+-----------+-------------+----------+------------+------------+


When it comes to all metrics, the numbers are pretty close, but we see that scikit-learn achieves a bit better results.



### XBoost

In [36]:
from xgboost import XGBClassifier

resultsxdepth = []

# Evaluate performance with different tree depths
for depth in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    xgb_model = XGBClassifier(n_estimators=50, learning_rate=0.1, max_depth=depth, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)
    
    # Train accuracy
    y_pred_train = xgb_model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_pred_train)
    
    # Test accuracy
    y_pred_test = xgb_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    resultsxdepth.append([depth, train_accuracy, test_accuracy])

headers = ["Max Depth", "Train Accuracy", "Test Accuracy"]
print(tabulate(resultsxdepth, headers=headers, tablefmt="grid"))


+-------------+------------------+-----------------+
|   Max Depth |   Train Accuracy |   Test Accuracy |
+=============+==================+=================+
|           1 |         0.788412 |        0.767347 |
+-------------+------------------+-----------------+
|           2 |         0.8244   |        0.794898 |
+-------------+------------------+-----------------+
|           3 |         0.84635  |        0.809184 |
+-------------+------------------+-----------------+
|           4 |         0.86779  |        0.832653 |
+-------------+------------------+-----------------+
|           5 |         0.885145 |        0.844898 |
+-------------+------------------+-----------------+
|           6 |         0.919091 |        0.852041 |
+-------------+------------------+-----------------+
|           7 |         0.945891 |        0.857143 |
+-------------+------------------+-----------------+
|           8 |         0.969627 |        0.861224 |
+-------------+------------------+------------

In [ ]:
resultsxlr = []

# Evaluate performance with different lr
for lr in [0.0001, 0.001, 0.01, 0.1, 0.2, 0.5, 1, 1.5,5,10]:
    xgb_model = XGBClassifier(n_estimators=50, learning_rate=lr, max_depth=4, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)
    
    # Train accuracy
    y_pred_train = xgb_model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_pred_train)
    
    # Test accuracy
    y_pred_test = xgb_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    resultsxlr.append([lr, train_accuracy, test_accuracy])

headers = ["Learning Rate", "Train Accuracy", "Test Accuracy"]
print(tabulate(resultsxlr, headers=headers, tablefmt="grid"))


+-----------------+------------------+-----------------+
|   Learning Rate |   Train Accuracy |   Test Accuracy |
+=================+==================+=================+
|          0.0001 |         0.787392 |        0.768367 |
+-----------------+------------------+-----------------+
|          0.001  |         0.787392 |        0.768367 |
+-----------------+------------------+-----------------+
|          0.01   |         0.787392 |        0.768367 |
+-----------------+------------------+-----------------+
|          0.1    |         0.86779  |        0.832653 |
+-----------------+------------------+-----------------+
|          0.2    |         0.887187 |        0.834694 |
+-----------------+------------------+-----------------+
|          0.5    |         0.941552 |        0.839796 |
+-----------------+------------------+-----------------+
|          1      |         0.985707 |        0.856122 |
+-----------------+------------------+-----------------+
|          1.5    |         0.9

In [ ]:
resultsxnest = []
# Evaluate performance with different number of trees
for n_estimatos in [10,30,50,70,100,130, 150]:
    xgb_model = XGBClassifier(n_estimators=n_estimatos, learning_rate=0.2, max_depth=4, eval_metric='logloss')
    xgb_model.fit(X_train, y_train)
    
    # Train accuracy
    y_pred_train = xgb_model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_pred_train)
    
    # Test accuracy
    y_pred_test = xgb_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    
    resultsxnest.append([n_estimatos, train_accuracy, test_accuracy])

headers = ["Number of Estimators", "Train Accuracy", "Test Accuracy"]
print(tabulate(resultsxnest, headers=headers, tablefmt="grid"))


+------------------------+------------------+-----------------+
|   Number of Estimators |   Train Accuracy |   Test Accuracy |
+========================+==================+=================+
|                     10 |         0.84048  |        0.804082 |
+------------------------+------------------+-----------------+
|                     30 |         0.868811 |        0.830612 |
+------------------------+------------------+-----------------+
|                     50 |         0.887187 |        0.834694 |
+------------------------+------------------+-----------------+
|                     70 |         0.904543 |        0.833673 |
+------------------------+------------------+-----------------+
|                    100 |         0.923941 |        0.840816 |
+------------------------+------------------+-----------------+
|                    130 |         0.944615 |        0.841837 |
+------------------------+------------------+-----------------+
|                    150 |         0.956

From the tables above we see that the best parameter choice would be depth 4, lr 0.1 or 0.2 and number of trees 50. We see the results for this combination in the last table, Train Accuracy: 0.887 Test Accuracy: 0.834. XBoost with this combination outperformes my custom model with train set where Train accuracy is 0.886 but for the test set the accuracy is same as mine.